In [2]:
!pip -q install gradio scikit-learn pandas numpy joblib

In [ ]:

# ============================================================
# 🌱 ADVANCED GUI - EXPLAINABLE AI CROP RECOMMENDATION
# Dataset download + ML training + Gradio GUI
# ============================================================

import os
import warnings
import joblib
import numpy as np
import pandas as pd
import gradio as gr

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 1. AUTOMATICALLY DOWNLOAD THE DATASET
# ------------------------------------------------------------

DATA_URL = (
    "https://raw.githubusercontent.com/"
    "ravikant-diwakar/AgriSens/master/"
    "Datasets/Crop_recommendation.csv"
)

print("Downloading crop recommendation dataset...")

try:
    df = pd.read_csv(DATA_URL)
except Exception as e:
    raise RuntimeError(
        "Could not download the dataset. Check your internet "
        "connection and try running this cell again."
    ) from e

df.columns = [str(c).strip().lower() for c in df.columns]

required = [
    "n", "p", "k", "temperature",
    "humidity", "ph", "rainfall", "label"
]

missing = [c for c in required if c not in df.columns]

if missing:
    raise ValueError(
        f"Dataset is missing columns: {missing}. "
        f"Available columns: {list(df.columns)}"
    )

df = df[required].copy()

feature_columns = [
    "n", "p", "k", "temperature",
    "humidity", "ph", "rainfall"
]

for col in feature_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["label"] = df["label"].astype(str).str.strip()
df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna(subset=required)
df = df[df["label"] != ""]

if df.empty:
    raise ValueError("No valid rows found in the downloaded dataset.")

print("Dataset downloaded successfully!")
print("Dataset size:", df.shape)
print("Available crops:", df["label"].nunique())
display(df.head())

# ------------------------------------------------------------
# 2. PREPARE DATA
# ------------------------------------------------------------

X = df[feature_columns]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# ------------------------------------------------------------
# 3. TRAIN THE MODEL
# ------------------------------------------------------------

print("Training the Random Forest model...")

best_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

best_model.fit(X_train, y_train)

print("Training completed!")

# ------------------------------------------------------------
# 4. EVALUATE MODEL PERFORMANCE
# ------------------------------------------------------------

predictions = best_model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

print(f"\nModel accuracy: {accuracy * 100:.2f}%")
print("\nClassification report:")
print(classification_report(y_test, predictions, zero_division=0))

# ------------------------------------------------------------
# 5. SAVE MODEL FILES
# ------------------------------------------------------------

joblib.dump(best_model, "/content/best_model.pkl")
joblib.dump(feature_columns, "/content/feature_columns.pkl")

medians = X_train.median().to_dict()
joblib.dump(medians, "/content/feature_medians.pkl")

print("Model saved successfully!")

# ------------------------------------------------------------
# 6. EXPLAINABLE PREDICTION
# ------------------------------------------------------------

def recommend_crops(n, p, k, temperature, humidity, ph, rainfall):

    values = {
        "Nitrogen": (n, 0, 200),
        "Phosphorus": (p, 0, 200),
        "Potassium": (k, 0, 200),
        "Temperature": (temperature, -10, 60),
        "Humidity": (humidity, 0, 100),
        "Soil pH": (ph, 0, 14),
        "Rainfall": (rainfall, 0, 1000)
    }

    for name, (value, low, high) in values.items():
        if value is None or not np.isfinite(float(value)):
            raise gr.Error(f"Enter a valid value for {name}.")

        if not low <= float(value) <= high:
            raise gr.Error(
                f"{name} must be between {low} and {high}."
            )

    input_data = pd.DataFrame([{
        "n": n,
        "p": p,
        "k": k,
        "temperature": temperature,
        "humidity": humidity,
        "ph": ph,
        "rainfall": rainfall
    }], columns=feature_columns)

    probabilities = best_model.predict_proba(input_data)[0]
    classes = best_model.classes_

    # Get the three highest model scores
    ranked = np.argsort(probabilities)[::-1][:3]

    top_results = pd.DataFrame([
        {
            "Crop": str(classes[i]).title(),
            "Model score (%)": round(float(probabilities[i]) * 100, 2)
        }
        for i in ranked
    ])

    best_index = int(np.argmax(probabilities))
    best_crop = str(classes[best_index]).title()
    confidence = float(probabilities[best_index]) * 100

    # Simple explanation using feature importance
    medians = joblib.load("/content/feature_medians.pkl")
    importance = best_model.feature_importances_

    values_by_feature = {
        "n": float(n),
        "p": float(p),
        "k": float(k),
        "temperature": float(temperature),
        "humidity": float(humidity),
        "ph": float(ph),
        "rainfall": float(rainfall)
    }

    names = {
        "n": "Nitrogen",
        "p": "Phosphorus",
        "k": "Potassium",
        "temperature": "Temperature",
        "humidity": "Humidity",
        "ph": "Soil pH",
        "rainfall": "Rainfall"
    }

    lines = [
        f"## 🌱 Recommended crop: {best_crop}",
        f"**Model score:** {confidence:.2f}%",
        "",
        "### 🔍 Explanation of your inputs",
        "The following features are important to the model overall:"
    ]

    order = np.argsort(importance)[::-1]

    for i in order[:4]:
        feature = feature_columns[i]
        current = values_by_feature[feature]
        median = float(medians[feature])

        if current > median:
            comparison = "above"
        elif current < median:
            comparison = "below"
        else:
            comparison = "equal to"

        lines.append(
            f"- **{names[feature]}:** {current:g}; "
            f"{comparison} the training-data median "
            f"({median:.2f}). Global feature importance: "
            f"{importance[i] * 100:.1f}%."
        )

    lines.extend([
        "",
        "**Note:** These comparisons describe your inputs; "
        "they do not prove why the model selected a particular crop. "
        "Model scores are not guaranteed real-world success rates."
    ])

    return best_crop, f"{confidence:.2f}%", top_results, "\n".join(lines)

# ------------------------------------------------------------
# 7. RESET FUNCTION
# ------------------------------------------------------------

def reset_app():
    return (
        50,   # Nitrogen
        50,   # Phosphorus
        50,   # Potassium
        25,   # Temperature
        80,   # Humidity
        200,  # Rainfall
        6.5,  # pH
        "",   # Recommended crop
        "",   # Model score
        pd.DataFrame(columns=["Crop", "Model score (%)"]),
        ""    # Explanation
    )

# ------------------------------------------------------------
# 8. CREATE GRADIO INTERFACE
# ------------------------------------------------------------

with gr.Blocks(
    theme=gr.themes.Soft(),
    title="Explainable AI Crop Recommendation"
) as demo:

    gr.Markdown("""
    # 🌱 Advanced GUI – Explainable AI Crop Recommendation

    Get crop recommendations based on soil nutrients and weather
    conditions using a trained Random Forest machine learning model.
    """)

    with gr.Row():
        with gr.Column():
            gr.Markdown("### 🌾 Soil Nutrients")

            n_input = gr.Slider(
                0, 200, value=50, step=1,
                label="Nitrogen (N)"
            )
            p_input = gr.Slider(
                0, 200, value=50, step=1,
                label="Phosphorus (P)"
            )
            k_input = gr.Slider(
                0, 200, value=50, step=1,
                label="Potassium (K)"
            )
            ph_input = gr.Slider(
                0, 14, value=6.5, step=0.1,
                label="Soil pH"
            )

        with gr.Column():
            gr.Markdown("### 🌦️ Weather Conditions")

            temp_input = gr.Slider(
                -10, 60, value=25, step=0.5,
                label="Temperature (°C)"
            )
            humidity_input = gr.Slider(
                0, 100, value=80, step=1,
                label="Humidity (%)"
            )
            rainfall_input = gr.Slider(
                0, 1000, value=200, step=1,
                label="Rainfall (mm)"
            )

    with gr.Row():
        predict_button = gr.Button(
            "🌱 Recommend Crops",
            variant="primary"
        )
        reset_button = gr.Button("🔄 Reset")

    gr.Markdown("## 📊 Recommendation Results")

    crop_output = gr.Textbox(
        label="Recommended Crop",
        interactive=False
    )
    score_output = gr.Textbox(
        label="Model Score",
        interactive=False
    )

    top_output = gr.Dataframe(
        headers=["Crop", "Model score (%)"],
        label="Top 3 Recommendations",
        interactive=False
    )

    explanation_output = gr.Markdown()

    predict_button.click(
        fn=recommend_crops,
        inputs=[
            n_input, p_input, k_input,
            temp_input, humidity_input,
            ph_input, rainfall_input
        ],
        outputs=[
            crop_output, score_output,
            top_output, explanation_output
        ]
    )

    reset_button.click(
        fn=reset_app,
        inputs=[],
        outputs=[
            n_input, p_input, k_input,
            temp_input, humidity_input, rainfall_input,
            ph_input, crop_output, score_output,
            top_output, explanation_output
        ]
    )

    gr.Markdown("""
    ---
    **Disclaimer:** Recommendations are based on the training dataset.
    Actual crop suitability depends on local soil testing, climate,
    season, irrigation and agricultural expertise.
    """)

# ------------------------------------------------------------
# 9. LAUNCH THE APP
# ------------------------------------------------------------

print("\nLaunching your application...")

demo.launch(share=True, debug=True)


Dataset downloaded successfully!
Dataset size: (2200, 8)
Available crops: 22


,n,p,k,temperature,humidity,ph,rainfall,label
0,90,42,43,20.879744,82.002744,6.502985,202.935536,rice
1,85,58,41,21.770462,80.319644,7.038096,226.655537,rice
2,60,55,44,23.004459,82.320763,7.840207,263.964248,rice
3,74,35,40,26.491096,80.158363,6.980401,242.864034,rice
4,78,42,42,20.130175,81.604873,7.628473,262.717340,rice


Training the Random Forest model...
Training completed!

Model accuracy: 99.55%

Classification report:
              precision    recall  f1-score   support

       apple       1.00      1.00      1.00        20
      banana       1.00      1.00      1.00        20
   blackgram       1.00      0.95      0.97        20
    chickpea       1.00      1.00      1.00        20
     coconut       1.00      1.00      1.00        20
      coffee       1.00      1.00      1.00        20
      cotton       1.00      1.00      1.00        20
      grapes       1.00      1.00      1.00        20
        jute       0.95      1.00      0.98        20
 kidneybeans       1.00      1.00      1.00        20
      lentil       1.00      1.00      1.00        20
       maize       0.95      1.00      0.98        20
       mango       1.00      1.00      1.00        20
   mothbeans       1.00      1.00      1.00        20
    mungbean       1.00      1.00      1.00        20
   muskmelon       1.00      1.